# ArmanNN Training on Google Colab

This notebook trains ArmanNN on FineWeb-Edu using an A100 GPU.

**Instructions:**
1. Go to Runtime → Change runtime type → Select **A100 GPU**
2. Run all cells in order

## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/mhd-rahman/Arman-NN.git
%cd Arman-NN

In [ ]:
!pip install -q torch datasets transformers numpy flash-attn --no-build-isolation

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load the dataset

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import IterableDataset, DataLoader

SEQ_LEN = 1024
HF_TOKEN = None  # Set your HuggingFace token here if needed, e.g. "hf_xxxx"
tokenizer = AutoTokenizer.from_pretrained("gpt2")

class StreamingTextDataset(IterableDataset):
    def __init__(self, dataset_name, subset, split, tokenizer, seq_len, token=None):
        self.dataset_name = dataset_name
        self.subset = subset
        self.split = split
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.token = token

    def __iter__(self):
        ds = load_dataset(self.dataset_name, self.subset, split=self.split,
                          streaming=True, token=self.token)
        buffer = []
        for example in ds:
            text = example["text"]
            if not text.strip():
                continue
            tokens = self.tokenizer.encode(text, add_special_tokens=False)
            buffer.extend(tokens)
            while len(buffer) >= self.seq_len + 1:
                chunk = buffer[:self.seq_len + 1]
                buffer = buffer[self.seq_len:]
                x = torch.tensor(chunk[:-1], dtype=torch.long)
                y = torch.tensor(chunk[1:], dtype=torch.long)
                yield x, y

train_dataset = StreamingTextDataset(
    "HuggingFaceFW/fineweb-edu", "sample-10BT", "train", tokenizer, SEQ_LEN, token=HF_TOKEN
)

# For eval — grab 5000 sequences from the stream
print("Loading eval sequences...")
eval_samples = []
for i, (x, y) in enumerate(train_dataset):
    eval_samples.append((x, y))
    if i >= 4999:
        break

class ListDataset(torch.utils.data.Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return self.samples[idx]

eval_dataset = ListDataset(eval_samples)
print(f"Eval: {len(eval_dataset)} sequences")
print("Streaming train dataset ready")

## 3. Configure and build the model

In [ ]:
from arman.model import ArmanConfig, ArmanNN

config = ArmanConfig(
    vocab_size=50257,       # GPT-2 tokenizer vocab
    d_model=1024,
    n_layers=12,
    n_heads=16,
    max_seq_len=1024,
    mlp_hidden=2816,
    expert_hidden=1792,
    n_experts=4,
    moe_top_k=2,
    ssm_state_size=128,
    use_graph=False,        # Disabled — not feeding graph data during training
    use_memory=False,       # Disabled — adds noise without structured input
)

model = ArmanNN(config)
print(f"Parameters: {model.parameter_count():,}")

## 4. Train with the Trainer

In [ ]:
import logging
import time
from pathlib import Path

# Clear any stale GPU memory from previous runs
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Force logging output to display in notebook cells
log = logging.getLogger()
log.setLevel(logging.INFO)
log.handlers = []
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
log.addHandler(handler)

from arman.training.scheduler import get_cosine_schedule_with_warmup
from arman.training.checkpointing import save_checkpoint, load_checkpoint, find_latest_checkpoint
from arman.training.evaluator import Evaluator, EvalConfig

# Training hyperparameters (H200 141GB)
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0
BATCH_SIZE = 64
GRAD_ACCUM = 2        # Effective batch = 128 sequences × 1024 tokens = ~131k tokens/step
WARMUP_STEPS = 0
TOTAL_STEPS = 76000
MIN_LR_RATIO = 0.1
SAVE_EVERY = 2000
EVAL_EVERY = 2000
LOG_EVERY = 50
CHECKPOINT_DIR = "checkpoints"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# model.enable_gradient_checkpointing()  # Saves ~60% memory, ~30% slower
print(f"Device: {device}")
print(f"Gradient checkpointing: enabled")

# Optimizer with weight decay separation
decay_params = [p for n, p in model.named_parameters() if p.requires_grad and p.ndim >= 2 and "norm" not in n]
no_decay_params = [p for n, p in model.named_parameters() if p.requires_grad and (p.ndim < 2 or "norm" in n)]
optimizer = torch.optim.AdamW([
    {"params": decay_params, "weight_decay": WEIGHT_DECAY},
    {"params": no_decay_params, "weight_decay": 0.0},
], lr=LEARNING_RATE, betas=(0.9, 0.95))

# Resume from checkpoint if available
global_step = 0
ckpt = find_latest_checkpoint(CHECKPOINT_DIR)
if ckpt:
    info = load_checkpoint(ckpt, model, optimizer=optimizer, scheduler=None, device=device)
    global_step = info["step"]
    print(f"Resumed from {ckpt} at step {global_step}")

scheduler = get_cosine_schedule_with_warmup(optimizer, WARMUP_STEPS, TOTAL_STEPS, MIN_LR_RATIO)
# Fast-forward scheduler to current step on resume
for _ in range(global_step):
    scheduler.step()

# bf16 — no GradScaler needed (more stable than fp16)
AMP_DTYPE = torch.bfloat16

# DataLoader for streaming dataset
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=4, pin_memory=True)

# Training loop
model.train()
optimizer.zero_grad(set_to_none=True)
data_iter = iter(train_loader)
step_start = time.time()

print(f"Training for {TOTAL_STEPS} steps | effective batch = {BATCH_SIZE * GRAD_ACCUM} | tokens/step = {BATCH_SIZE * GRAD_ACCUM * SEQ_LEN:,}")
print(f"Starting from step {global_step}")

while global_step < TOTAL_STEPS:
    accum_loss = 0.0
    accum_aux = 0.0

    for _ in range(GRAD_ACCUM):
        try:
            x, y = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            x, y = next(data_iter)

        x, y = x.to(device), y.to(device)
        with torch.amp.autocast("cuda", dtype=AMP_DTYPE):
            out = model(x, targets=y)
            loss = out["loss"] / GRAD_ACCUM

        loss.backward()
        accum_loss += out["loss"].item() / GRAD_ACCUM
        accum_aux += out["aux_loss"].item() / GRAD_ACCUM

    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    scheduler.step()
    global_step += 1

    # Log
    if global_step % LOG_EVERY == 0:
        dt = time.time() - step_start
        lr = scheduler.get_last_lr()[0]
        print(f"step={global_step:06d} | loss={accum_loss:.4f} | aux={accum_aux:.4f} | "
              f"grad_norm={grad_norm:.3f} | lr={lr:.2e} | dt={dt:.2f}s")
        step_start = time.time()

    # Checkpoint
    if global_step % SAVE_EVERY == 0:
        ckpt_path = Path(CHECKPOINT_DIR) / f"step_{global_step:06d}.pt"
        save_checkpoint(ckpt_path, model, optimizer, scheduler, global_step, config)
        print(f"Saved checkpoint: {ckpt_path}")

    # Eval
    if global_step % EVAL_EVERY == 0:
        eval_cfg = EvalConfig(batch_size=16, use_amp=True, amp_dtype="bfloat16")
        evaluator = Evaluator(model=model, eval_config=eval_cfg, device=device)
        metrics = evaluator.evaluate(eval_dataset)
        print(f"[eval] step={global_step:06d} | {metrics}")
        model.train()

# Final save
ckpt_path = Path(CHECKPOINT_DIR) / f"step_{global_step:06d}.pt"
save_checkpoint(ckpt_path, model, optimizer, scheduler, global_step, config)
print(f"Training complete! Final checkpoint: {ckpt_path}")

## 5. Evaluate the trained model

In [ ]:
from arman.training import Evaluator, EvalConfig

eval_config = EvalConfig(batch_size=16, use_amp=True, amp_dtype="bfloat16")
evaluator = Evaluator(model=model, eval_config=eval_config, device=device)

metrics = evaluator.evaluate(eval_dataset)
print("\n" + "=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  Loss:           {metrics.loss:.4f}")
print(f"  Perplexity:     {metrics.perplexity:.2f}")
print(f"  Top-1 Accuracy: {metrics.top1_accuracy*100:.2f}%")
print(f"  Top-5 Accuracy: {metrics.top5_accuracy*100:.2f}%")
print(f"  MRR:            {metrics.mrr:.4f}")
print("=" * 50)

## 6. Generate text from the trained model

In [ ]:
from transformers import AutoTokenizer
from generate import generate

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Encode a prompt
prompt = "The most important concept in machine learning is"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

# Generate
model.eval()
output_ids = generate(
    model,
    input_ids,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50,
    top_p=0.9,
)

# Decode
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"Generated: {generated_text}")

## 7. Save checkpoint to Google Drive (optional)

In [ ]:
# Mount Google Drive and copy checkpoint for persistence
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/ArmanNN
!cp -r checkpoints/ /content/drive/MyDrive/ArmanNN/
print("Checkpoints saved to Google Drive!")